# 1)VectorStoreRetriever	Vector-based	Embedding similarity search	General-purpose RAG

# ConversationalRetrievalChain

# WITH AGENT MODE

# RUN V/S INVOKE

# RAG MEANS WE ARE CREATING A QUERY AND WE FETCH INFORMATION THAT IS REALTED TO QUERY AND THEN PASS IT TO LLM 
# TO GROUND THE RESPONSE WTHIN THAT QUERY AND CONTEXT

# that is why we used retriever in chain.run/invoke with llm instaed of prompt

In [ ]:
import os
from langchain.chat_models import ChatOpenAI
from langchain.chains import ConversationalRetrievalChain
from langchain.tools import StructuredTool # importing structured tool for passing invoke instead of run
from langchain.agents import initialize_agent, AgentType
from langchain.memory import ConversationBufferMemory
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

In [ ]:
# Load keys
from dotenv import load_dotenv

# 2️⃣ Load API keys
load_dotenv(".env")
openai_api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
# initialize llm
llm = ChatOpenAI(model="gpt-4o-mini",temperature=0)

In [ ]:
# 2. Vector DB
with open("sample.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

In [ ]:
text_data

In [ ]:
# 🧠 Split the text into smaller chunks
from langchain.text_splitter import CharacterTextSplitter

splitter = CharacterTextSplitter(separator="\n", chunk_size=20, chunk_overlap=10)
texts = splitter.split_text(text_data)

In [ ]:
texts

The "Static" vs "Dynamic" Retrieval
Because you created the FAISS index using FAISS.from_texts(texts, embedding), the database is static for the duration of the script.
The retriever is searching through the chunks of sample.txt you provided at the start.

In [ ]:
embedding = OpenAIEmbeddings()
vectorstore = FAISS.from_texts(texts, embedding)
retriever = vectorstore.as_retriever()

In [ ]:
# 3. Conversational RAG chain

# In chain you pass the llm and the prompt, here you pass the llm and retriever

rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    return_source_documents=False
)

In [ ]:
# 4. Memory for chat history
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

In [ ]:
# 5. Wrap RAG as a StructuredTool
def rag_tool_fn(question: str) -> str:
    return rag_chain.invoke({"question": question,"chat_history":[]})["answer"] # simply setting chat history as blank

rag_tool = StructuredTool.from_function(
    name="RAG_QA",
    description="Use this to answer questions about LangChain.",
    func=rag_tool_fn
)

In [ ]:
# 6. Create agent
agent = initialize_agent(
    tools=[rag_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    memory=memory,
    handle_parsing_errors=True
)

In [ ]:
# 7. Run conversation
print("1️⃣ First question")
res1 = agent.run("What is LangChain?")
print("Answer:", res1)

print("\n2️⃣ Follow-up")
res2 = agent.run("Who created it?")
print("Answer:", res2)

print("\n3️⃣ Ask again")
res3 = agent.run("Explain LangChain again simply.")
print("Answer:", res3)

In [ ]:
res1

In [ ]:
res2

In [ ]:
res3

# 7. invoke conversation
print("1️⃣ First question")
res1 = agent.invoke({"input":"What is LangChain?"})
print("Answer:", res1)

print("\n2️⃣ Follow-up")
res2 = agent.invoke({"input":"Who created it?"})
print("Answer:", res2)

print("\n3️⃣ Ask again")
res3 = agent.invoke({"input":"Explain LangChain again simply."})
print("Answer:", res3)

In [ ]:
print(agent.input_keys)

# USING INVOKE METHOD INSTEAD OF RUN

In [1]:
#1)VectorStoreRetriever	Vector-based	Embedding similarity search	General-purpose RAG
# ConversationalRetrievalChain

from langchain.chat_models import ChatOpenAI
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import ConversationalRetrievalChain
from langchain.agents import initialize_agent, AgentType
from langchain.tools import StructuredTool
from langchain.memory import ConversationBufferMemory
import os

# 1. Setup

import os
from dotenv import load_dotenv

# 2️⃣ Load API keys
load_dotenv(".env")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


llm = ChatOpenAI(temperature=0)

# 2. Vector DB
with open("sample.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

# 🧠 Split the text into smaller chunks
from langchain.text_splitter import CharacterTextSplitter
splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
texts = splitter.split_text(text_data)

embedding = OpenAIEmbeddings()
vectorstore = FAISS.from_texts(texts, embedding)
retriever = vectorstore.as_retriever()

# 3. Conversational RAG chain
rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    return_source_documents=False
)

# 4. Memory for chat history
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# 5. Wrap RAG as a StructuredTool
def rag_tool_fn(question) :
    return rag_chain.invoke({
        "question": question,
        "chat_history": []
    })["answer"]

rag_tool = StructuredTool.from_function(
    name="RAG_QA",
    description="Use this to answer questions about LangChain.",
    func=rag_tool_fn
)

# 6. Create agent
agent = initialize_agent(
    tools=[rag_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    memory=memory,
    handle_parsing_errors=True
)

# 7. invoke conversation
print("1️⃣ First question")
res1 = agent.invoke({"input":"What is LangChain?"})
print("Answer:", res1.get('output'))
print(res1)

print("\n2️⃣ Follow-up")
res2 = agent.invoke({"input":"Who created it?"})
print("Answer:", res2.get('output'))

print("\n3️⃣ Ask again")
res3 = agent.invoke({"input":"Explain LangChain again simply."})
print("Answer:", res3.get('output'))


C:\Users\my pc\AppData\Local\Temp\ipykernel_4776\3221048821.py:23: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(temperature=0)
C:\Users\my pc\AppData\Local\Temp\ipykernel_4776\3221048821.py:34: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embedding = OpenAIEmbeddings()
C:\Users\my pc\AppData\Local\Temp\ipykernel_4776\3221048821.py:46: LangChainDeprecationWarning

1️⃣ First question


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: RAG_QA
Action Input: What is LangChain?
Observation: LangChain is a framework created by Harrison Chase for building applications with Large Language Models (LLMs). It supports various features such as RAG, agents, memory, tools, and more. LangChain is commonly used in applications like chatbots, document Q&A, and AI workflow.
Thought:Do I need to use a tool? No
AI: LangChain is a framework created by Harrison Chase for building applications with Large Language Models (LLMs). It supports various features such as RAG, agents, memory, tools, and more. LangChain is commonly used in applications like chatbots, document Q&A, and AI workflow.

> Finished chain.
Answer: LangChain is a framework created by Harrison Chase for building applications with Large Language Models (LLMs). It supports various features such as RAG, agents, memory, tools, and more. LangChain is commonly used in applic

In [ ]:
res1.get('output')

In [2]:
res1

{'input': 'What is LangChain?',
 'chat_history': [HumanMessage(content='What is LangChain?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='LangChain is a framework created by Harrison Chase for building applications with Large Language Models (LLMs). It supports various features such as RAG, agents, memory, tools, and more. LangChain is commonly used in applications like chatbots, document Q&A, and AI workflow.', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Who created it?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='LangChain was created by Harrison Chase.', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Explain LangChain again simply.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='LangChain is a framework created by Harrison Chase for building applications using Large Language Models (LLMs). It supports various features like RAG, agents, memory, tools, and more. LangChain is 

In [3]:
res2

{'input': 'Who created it?',
 'chat_history': [HumanMessage(content='What is LangChain?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='LangChain is a framework created by Harrison Chase for building applications with Large Language Models (LLMs). It supports various features such as RAG, agents, memory, tools, and more. LangChain is commonly used in applications like chatbots, document Q&A, and AI workflow.', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Who created it?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='LangChain was created by Harrison Chase.', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Explain LangChain again simply.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='LangChain is a framework created by Harrison Chase for building applications using Large Language Models (LLMs). It supports various features like RAG, agents, memory, tools, and more. LangChain is com

In [4]:
res3

{'input': 'Explain LangChain again simply.',
 'chat_history': [HumanMessage(content='What is LangChain?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='LangChain is a framework created by Harrison Chase for building applications with Large Language Models (LLMs). It supports various features such as RAG, agents, memory, tools, and more. LangChain is commonly used in applications like chatbots, document Q&A, and AI workflow.', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Who created it?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='LangChain was created by Harrison Chase.', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Explain LangChain again simply.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='LangChain is a framework created by Harrison Chase for building applications using Large Language Models (LLMs). It supports various features like RAG, agents, memory, tools, and more. 

In [ ]:
#1)VectorStoreRetriever	Vector-based	Embedding similarity search	General-purpose RAG
# ConversationalRetrievalChain

from langchain.chat_models import ChatOpenAI
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import ConversationalRetrievalChain
from langchain.agents import initialize_agent, AgentType
from langchain.tools import StructuredTool
from langchain.memory import ConversationBufferMemory
import os

# 1. Setup

import os
from dotenv import load_dotenv

# 2️⃣ Load API keys
load_dotenv(".env")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


llm = ChatOpenAI(temperature=0)

# 2. Vector DB
with open("sample.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

# 🧠 Split the text into smaller chunks
from langchain.text_splitter import CharacterTextSplitter
splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
texts = splitter.split_text(text_data)

embedding = OpenAIEmbeddings()
vectorstore = FAISS.from_texts(texts, embedding)
retriever = vectorstore.as_retriever()

# 3. Conversational RAG chain
rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    return_source_documents=False
)

# 4. Memory for chat history
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# 5. Wrap RAG as a StructuredTool
def rag_tool_fn(question) :
    return rag_chain.invoke({
        "question": question,
        "chat_history": []
    })["answer"]

rag_tool = StructuredTool.from_function(
    name="RAG_QA",
    description="Use this to answer questions about LangChain.",
    func=rag_tool_fn
)

# 6. Create agent
agent = initialize_agent(
    tools=[rag_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    memory=memory,
    handle_parsing_errors=True
)

# 7. Run conversation
print("1️⃣ First question")
res1 = agent.run("What is LangChain?")
print("Answer:", res1)

print("\n2️⃣ Follow-up")
res2 = agent.run("Who created it?")
print("Answer:", res2)

print("\n3️⃣ Ask again")
res3 = agent.run("Explain LangChain again simply.")
print("Answer:", res3)

In [ ]:
res1